In [1]:
import getpass
import json
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from anthropic import Anthropic

# Securely prompt for the API key string
api_key = getpass.getpass("Enter your Anthropic API Key: ")
client = Anthropic(api_key=api_key)

print("✅ Initialization complete. Anthropic client and LangGraph dependencies ready.")

✅ Initialization complete. Anthropic client and LangGraph dependencies ready.


In [2]:
# 1. DEFINE THE STATE SCHEMA
class PipelineState(TypedDict):
    step_history: list
    is_recovered: bool

# 2. DEFINE THE NODES
def build_node(state: PipelineState) -> dict:
    print("\n📦 [NODE -> BUILD]: Compiling source code and building container image...")
    current_history = state.get("step_history", []) + ["BUILD_SUCCESS"]
    return {"step_history": current_history}

def deploy_node(state: PipelineState) -> dict:
    print("\n🚀 [NODE -> DEPLOY]: Attempting deployment to Staging environment...")
    
    # Intentional Bug / Crash point
    if not state.get("is_recovered", False):
        print("❌ [CRITICAL ERROR -> DEPLOY]: Connection timeout while pushing image to registry!")
        print("!!! APPLICATION CRASHED !!!")
        raise ConnectionError("Registry unavailable")
        
    print("✅ [NODE -> DEPLOY]: Image pushed and pods deployed successfully to Staging!")
    current_history = state.get("step_history", []) + ["DEPLOY_SUCCESS"]
    return {"step_history": current_history}

def smoke_test_node(state: PipelineState) -> dict:
    print("\n🧪 [NODE -> SMOKE TEST]: Running automated API test suite against Staging...")
    current_history = state.get("step_history", []) + ["SMOKE_TEST_PASSED"]
    return {"step_history": current_history}

In [3]:
# 3. BUILD THE GRAPH LOGIC
builder = StateGraph(PipelineState)
builder.add_node("build", build_node)
builder.add_node("deploy", deploy_node)
builder.add_node("smoke_test", smoke_test_node)

builder.add_edge(START, "build")
builder.add_edge("build", "deploy")
builder.add_edge("deploy", "smoke_test")
builder.add_edge("smoke_test", END)

# Attach the memory checkpointer ("Save Game" engine)
memory_checkpointer = MemorySaver()
compiled_pipeline = builder.compile(checkpointer=memory_checkpointer)

print("🎯 Pipeline graph compiled successfully with persistent checkpointing attached.")

🎯 Pipeline graph compiled successfully with persistent checkpointing attached.


In [5]:
# Configure our tracking identifier session
config = {"configurable": {"thread_id": "interactive_lab_run"}}

# =====================================================================
# PHASE 1: TRIGGER THE FIRST PIPELINE EXECUTION (EXPECT CRASH)
# =====================================================================
print("=" * 70)
print("PHASE 1: RUNNING INITIAL PIPELINE EXECUTION")
print("=" * 70)

try:
    initial_input = {"step_history": [], "is_recovered": False}
    # Use streaming to walk through updates cleanly
    for event in compiled_pipeline.stream(initial_input, config=config, stream_mode="updates"):
        for node_name, state_update in event.items():
            print(f"🔹 Finished node '{node_name}'. Update: {state_update}")
            # Step Pause after successful node execution
            input(f"\n[PAUSED] Node '{node_name}' finished. Press Enter to continue pipeline...")
except Exception as e:
    print(f"\n🛑 Pipeline interrupted by exception: {str(e)}")
    print("Execution halted. Memory state is preserved in the checkpointer database.")


# =====================================================================
# PHASE 2: INSPECT SNAPSHOTS & CALL CLAUDE FOR ANALYSIS
# =====================================================================
print("\n" + "=" * 70)
print("PHASE 2: INSPECTING DATABASE HISTORY SNAPSHOTS WITH CLAUDE")
print("=" * 70)

# Read historical state checkpoints directly out of the database using the Thread ID
history = list(compiled_pipeline.get_state_history(config))
history_summary = []

print(f"Total saved snapshots found for thread '{config['configurable']['thread_id']}': {len(history)}")
for idx, snapshot in enumerate(reversed(history)):
    summary_str = f"Snapshot {idx} | Next Node to execute: {snapshot.next} | State Value: {snapshot.values}"
    print(summary_str)
    history_summary.append(summary_str)

# Pass the historical snapshot traces to Claude to explain the state situation
print("\n🤖 [Claude]: Reviewing ledger history snapshots...")
prompt = f"""Review these LangGraph thread snapshots captured after an application crash:
{chr(10).join(history_summary)}

Explain concisely:
1. Which step completed successfully?
2. At which node did the application crash?
3. What state value changes occurred before it went down?"""

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=500,
    messages=[{"role": "user", "content": prompt}]
)
print(f"\n[Claude Diagnostics Breakdown]:\n{response.content[0].text}")

# Wait for user permission before executing recovery actions
input("\n[PAUSED] Press Enter to ALLOW the agent to update the state values and trigger recovery...")


# =====================================================================
# PHASE 3: RECOVER AND RESUME STATE FROM THE EXACT FAULT POINT
# =====================================================================
print("\n" + "=" * 70)
print("PHASE 3: STATE RECOVERY & RESUMPTION")
print("=" * 70)

print("🛠️ [RECOVERY MODE]: Simulating fix by updating state variable 'is_recovered' to True...")
compiled_pipeline.update_state(config, {"is_recovered": True})

print("♻️ [RECOVERY MODE]: Booting graph with original thread context configuration...")
# Passing None as the input tells LangGraph to fetch the last save-point from the checkpointer database!
final_output = compiled_pipeline.invoke(None, config=config)

print("\n🎉 [FINAL RESULTS]: Run Complete!")
print(f"Final step history ledger: {final_output['step_history']}")

PHASE 1: RUNNING INITIAL PIPELINE EXECUTION

📦 [NODE -> BUILD]: Compiling source code and building container image...
🔹 Finished node 'build'. Update: {'step_history': ['BUILD_SUCCESS']}

🚀 [NODE -> DEPLOY]: Attempting deployment to Staging environment...
❌ [CRITICAL ERROR -> DEPLOY]: Connection timeout while pushing image to registry!
!!! APPLICATION CRASHED !!!

🛑 Pipeline interrupted by exception: Registry unavailable
Execution halted. Memory state is preserved in the checkpointer database.

PHASE 2: INSPECTING DATABASE HISTORY SNAPSHOTS WITH CLAUDE
Total saved snapshots found for thread 'interactive_lab_run': 9
Snapshot 0 | Next Node to execute: ('__start__',) | State Value: {}
Snapshot 1 | Next Node to execute: ('build',) | State Value: {'step_history': [], 'is_recovered': False}
Snapshot 2 | Next Node to execute: ('deploy',) | State Value: {'step_history': ['BUILD_SUCCESS'], 'is_recovered': False}
Snapshot 3 | Next Node to execute: ('deploy',) | State Value: {'step_history': ['BU